This will find all contractions for samples of a ball around manifold points
at layer 0

In [ ]:
# Import all basic functions
import onnx
import onnxruntime as ort
import ml_dtypes
import numpy as np
import h5py
import torch
import torch.utils.dlpack

# Import config
from config import load_config, InputSourceConfig

import importlib
import sys
import tqdm

DEVICE = torch.device("cuda")


In [ ]:
# data generation
from generate_data.datagen import (
    sample_ambient_points,
    sample_around_points,  # kept for reference; NOT used below, replaced by sample_around_points_gpu
    repeat_points_to_match,
    get_token_ids_array,
)

# model generation
from construct.construct import (
    ensure_base_model,
    ensure_subgraph_pullback,
    ensure_subgraphs,
)

# execution code
from execute.optimization_loop import optimization_loop, NewtonRaphson

# 1. Reload the parent module
importlib.reload(sys.modules["execute.optimization_loop"])
importlib.reload(sys.modules["generate_data.datagen"])
importlib.reload(sys.modules["construct.construct"])


In [ ]:
# --- IOBinding utilities ---
# NOTE: bfloat16 support through IOBinding's element_type has NOT been verified
# against your installed onnxruntime version. Run the smoke-test cell further
# down ("bfloat16 IOBinding smoke test") before trusting the main loop.

def _torch_to_np_dtype(dtype: torch.dtype):
    mapping = {
        torch.float32: np.float32,
        torch.float64: np.float64,
        torch.int64: np.int64,
        torch.int32: np.int32,
        torch.bfloat16: ml_dtypes.bfloat16,
    }
    if dtype not in mapping:
        raise ValueError(f"No numpy/ml_dtypes mapping registered for {dtype}")
    return mapping[dtype]


def run_iobinding_gpu(session, input_name, input_tensor, output_name, device_id=0):
    """
    Run an onnxruntime session with a GPU-resident torch input tensor,
    returning a GPU-resident torch output tensor.
    Uses IOBinding + dlpack to avoid host<->device copies on input and output.
    """
    input_tensor = input_tensor.contiguous()
    io_binding = session.io_binding()

    io_binding.bind_input(
        name=input_name,
        device_type="cuda",
        device_id=device_id,
        element_type=_torch_to_np_dtype(input_tensor.dtype),
        shape=tuple(input_tensor.shape),
        buffer_ptr=input_tensor.data_ptr(),
    )
    io_binding.bind_output(output_name, device_type="cuda", device_id=device_id)

    session.run_with_iobinding(io_binding)

    ort_output = io_binding.get_outputs()[0]
    return torch.utils.dlpack.from_dlpack(ort_output._ortvalue.to_dlpack())


def run_iobinding_cpu_input_gpu_output(session, input_name, input_array, output_name, device_id=0):
    """
    For host-side inputs (e.g. int64 token ids as plain numpy arrays).
    ORT copies the input host->device once internally; output stays
    GPU-resident as a torch tensor via dlpack.
    """
    io_binding = session.io_binding()
    io_binding.bind_cpu_input(input_name, input_array)
    io_binding.bind_output(output_name, device_type="cuda", device_id=device_id)

    session.run_with_iobinding(io_binding)

    ort_output = io_binding.get_outputs()[0]
    return torch.utils.dlpack.from_dlpack(ort_output._ortvalue.to_dlpack())


In [ ]:
# --- GPU replacement for sample_around_points ---

def sample_around_points_gpu(
    points: torch.Tensor,
    n_samples: int,
    min_radius: float = 0.0,
    max_radius: float = 1.0,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    device = points.device
    batch, seq, dim = points.shape

    directions = torch.randn(
        (n_samples, batch, seq, dim), device=device, dtype=torch.float32, generator=generator
    )
    directions /= directions.norm(dim=-1, keepdim=True)

    u = torch.rand((n_samples, batch, seq, 1), device=device, dtype=torch.float32, generator=generator)

    if min_radius == 0.0:
        radii = max_radius * u.pow(1.0 / dim)
    else:
        ratio = min_radius / max_radius
        radii = max_radius * (ratio**dim + u * (1.0 - ratio**dim)).pow(1.0 / dim)

    directions.mul_(radii)
    return directions.add_(points.unsqueeze(0))


In [ ]:
cfg = load_config("configs/qwen_jac_gather.toml")
force = False


# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)
sub_graphs = ensure_subgraphs(cfg, force=force)

# Define output session
sess_options = ort.SessionOptions()

# Very Important --- will cause errors if missing
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.intra_op_num_threads = 8  # pick a value <= your usable core count

print(sub_graphs[0][0])
embedding_gen = ort.InferenceSession(
    str(sub_graphs[0][0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

# Renamed from generic input_names/output_names (original code reused those
# names for out_sess below, silently overwriting them) and switched from sets
# to ordered lists matching get_outputs()/get_inputs() order -- this is the
# same order .run(None, ...) returns outputs in, and is required for IOBinding
# to bind the correct output.
emb_outputs_meta = embedding_gen.get_outputs()
emb_inputs_meta = embedding_gen.get_inputs()
emb_output_names = [o.name for o in emb_outputs_meta]
emb_input_names = [o.name for o in emb_inputs_meta]
print(emb_input_names, emb_output_names)


In [ ]:
# Load config
cfg = load_config("configs/qwen_jac_sublayer_tokens_new.toml")

# Re-use all existing models
force = False

# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)

force = True
# makes sure that out subgraphs are present
sub_graphs = ensure_subgraphs(cfg, force=force)

# makes sure that we have the jacobains
sub_graphs_pullbacks = ensure_subgraph_pullback(
    sub_graphs,
    force=force,
)


In [ ]:
out_sess = ort.InferenceSession(
    str(sub_graphs[0][0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

# Renamed + ordered lists -- see note in the embedding_gen cell above.
out_outputs_meta = out_sess.get_outputs()
out_inputs_meta = out_sess.get_inputs()
out_output_names = [o.name for o in out_outputs_meta]
out_input_names = [o.name for o in out_inputs_meta]

print(out_output_names, out_input_names)


In [ ]:
inputs = get_token_ids_array(
    cfg.input_source,
    cfg.seq,
)
new_input = np.repeat(
    inputs.reshape(1, *inputs.shape),
    cfg.vocab_size,
    axis=0,
)
new_input[:, :, -1] = np.arange(cfg.vocab_size).reshape(-1, 1)


In [ ]:
# x_out entries are now GPU-resident torch tensors (float32), not numpy arrays.
# input_ids stays host-side (bind_cpu_input, cheap: it's just token ids); the
# output is bound directly to GPU memory and retrieved via dlpack, avoiding
# the extra device->host->device round trip the original .astype(np.float32)
# on a numpy result implied.
gen = torch.Generator(device=DEVICE)
x_out = []
y_out = []
out_in_name = out_input_names[0]
out_out_name = out_output_names[0]
dx_list = []
dfx_list = []
for s in tqdm.tqdm(range(new_input.shape[0])):
    token_ids = new_input[s]
    x_bf16 = run_iobinding_cpu_input_gpu_output(
        embedding_gen, emb_input_names[0], token_ids, emb_output_names[0],
    )
    y0_bf16 = sample_around_points_gpu(x_bf16, n_samples=1000, max_radius=1e-3, generator=gen).to(torch.bfloat16)
    _fx = run_iobinding_gpu(out_sess, out_in_name, x_bf16, out_out_name)
    for i in range(y0_bf16.shape[0]):
        fy_s = run_iobinding_gpu(out_sess, out_in_name, y0_bf16[i], out_out_name).to(torch.float32)

        dx_list.append((y0_bf16[i] - x_bf16).norm(dim=-1))
        dfx_list.append((fy_s - _fx).norm(dim=-1))
    

What is the relative positioning of all of these points?

From the zero ids go back to vectors and then from there figure out how is this related to the kernel of the jacobian
